<a href="https://colab.research.google.com/github/ghn20uc/Project-Silvanus/blob/crawler/Map_DATA309.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
!pip install dash

Select and upload file NZ_FR_IAR_with_site_conditions and vs30_site_lookup from pipeline then run everything


In [ ]:
# @title
from google.colab import files

# upload file NZ_FR_IAR_with_site_conditions and vs30_site_lookup from pipeline then run everything
upload = files.upload()

Saving NZ_FR_IAR_with_site_conditions.parquet to NZ_FR_IAR_with_site_conditions (2).parquet
Saving vs30_site_lookup.parquet to vs30_site_lookup (2).parquet


In [ ]:
# @title
import dask.dataframe as dd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, dcc, html, Input, Output

In [ ]:
# @title
NZ_FR_IAR_with_site_conditions = dd.read_parquet(
    '/content/NZ_FR_IAR_with_site_conditions.parquet'
)

vs30_site_lookup = dd.read_parquet(
    '/content/vs30_site_lookup.parquet'
)

In [ ]:
# @title
def doc_col(file):
  for i, col in enumerate(file.columns):
    print(i, col)

In [ ]:
# @title
doc_col(NZ_FR_IAR_with_site_conditions)

0 publicid
1 cell_id
2 feature_index
3 report_longitude
4 report_latitude
5 reported_mmi
6 report_count
7 geohash
8 geometry_type
9 aggregation
10 eventtype
11 origintime
12 modificationtime
13 event_longitude
14 event_latitude
15 catalogue_magnitude
16 event_depth_km
17 magnitudetype
18 depthtype
19 evaluationmethod
20 evaluationstatus
21 evaluationmode
22 earthmodel
23 usedphasecount
24 usedstationcount
25 magnitudestationcount
26 minimumdistance
27 azimuthalgap
28 originerror
29 magnitudeuncertainty
30 source_mtime
31 year
32 tectonic_region
33 felt_query_eligible
34 felt_query_exclusion_reason
35 felt_query_status
36 felt_query_http_status
37 felt_api_feature_count
38 felt_observation_count
39 felt_query_attempts
40 felt_query_message
41 felt_aggregation
42 felt_queried_at_utc
43 felt_cell_count
44 felt_cells_with_mmi
45 felt_report_count
46 felt_mmi_min
47 felt_mmi_median_cells
48 felt_mmi_max
49 felt_mmi_weighted_mean
50 felt_mmi_weighted_median
51 felt_mmi_3_reports
52 felt_mmi_

In [ ]:
# @title
doc_col(vs30_site_lookup)

0 report_longitude
1 report_latitude
2 site_lookup_id
3 Vs30_raw
4 vs30_raster_cell
5 vs30_cell_x
6 vs30_cell_y
7 vs30_extraction_source
8 vs30_source_distance_m
9 vs30_plausible
10 Vs30_m_s
11 has_vs30
12 vs30_nearest_fill_used
13 ln_Vs30
14 ln_Vs30_over_760
15 site_class_vs30_proxy
16 site_class_broad
17 vs30_status


In [ ]:
# @title
data = NZ_FR_IAR_with_site_conditions

data['origintime'] = dd.to_datetime(data['origintime'])

In [ ]:
# @title
data.head()

,publicid,cell_id,feature_index,report_longitude,report_latitude,reported_mmi,report_count,geohash,geometry_type,aggregation,...,vs30_cell_y,vs30_source_distance_m,vs30_extraction_source,vs30_nearest_fill_used,vs30_status,ln_Vs30,ln_Vs30_over_760,site_class_vs30_proxy,site_class_broad,site_model_eligible
0,2012p001403,2012p001403_cell_52,52,172.721558,-43.481140,4.0,1,<NA>,Point,median,...,-43.481583,61.836604,containing_raster_cell,False,available_containing_cell,5.251686,-1.381632,D,soft_soil,True
1,2012p001403,2012p001403_cell_149,149,172.721558,-43.492126,4.0,8,<NA>,Point,median,...,-43.492377,46.593882,containing_raster_cell,False,available_containing_cell,5.319328,-1.313990,D,soft_soil,True
2,2012p001403,2012p001403_cell_117,117,172.732544,-43.508606,4.0,3,<NA>,Point,median,...,-43.508568,63.467505,nearest_nonmissing_raster_cell,True,available_nearest_cell,5.216089,-1.417230,D,soft_soil,True
3,2012p001403,2012p001403_cell_134,134,172.743530,-43.530579,4.0,2,<NA>,Point,median,...,-43.531056,285.565488,nearest_nonmissing_raster_cell,True,available_nearest_cell,5.179245,-1.454073,E,very_soft_soil,True
4,2012p001403,2012p001403_cell_107,107,172.699585,-43.481140,4.0,9,<NA>,Point,median,...,-43.481583,51.464199,containing_raster_cell,False,available_containing_cell,5.260558,-1.372760,D,soft_soil,True


In [ ]:
# @title
layer_0 = data[[
                'publicid',
                'origintime',
                'event_latitude',
                'event_longitude',
                'Mw_model',
                'event_depth_km',
                'fault_style'
                ]].drop_duplicates(subset = 'publicid').compute()

layer_1 = data[[
                'publicid',
                'origintime',
                'report_latitude',
                'report_longitude',
                'reported_mmi',
                'Vs30_m_s',
                'Repi_km',
                'Rhypo_km'
                ]]

layer_0['Mw_model'] = (
    layer_0['Mw_model']
    .astype(str)
    .str.strip()
)

layer_0['Mw_model'] = pd.to_numeric(
    layer_0['Mw_model'],
    errors='coerce'
)

layer_0['event_latitude'] = pd.to_numeric(
    layer_0['event_latitude'],
    errors='coerce'
)

layer_0['event_longitude'] = pd.to_numeric(
    layer_0['event_longitude'],
    errors='coerce'
)

layer_0 = layer_0.dropna(
    subset=[
        'event_latitude',
        'event_longitude',
        'Mw_model'
    ]
)

In [ ]:
# @title
print(layer_0[[
              'publicid',
              'origintime',
              'event_latitude',
              'event_longitude',
              'Mw_model',
              'event_depth_km',
              'fault_style'
              ]
              ].isna().sum()
      )

#fault has NA so no draw with fault

publicid            0
origintime          0
event_latitude      0
event_longitude     0
Mw_model            0
event_depth_km      0
fault_style        66
dtype: int64


In [ ]:
# @title
source_fig = px.scatter_map(
    layer_0,

    lat='event_latitude',
    lon='event_longitude',

    color='Mw_model',

    color_continuous_scale='Turbo',

    range_color=[3, 8],

    hover_name='publicid',

    hover_data=[
        'origintime',
        'Mw_model',
        'event_depth_km',
    ],

    custom_data=[
        'publicid'
    ],

    zoom=4.5,

    center={
        'lat': -41.3,
        'lon': 172.5
    },

    title='New Zealand Earthquake Sources',

    labels={
        'Mw_model': 'Magnitude (Mw)'
    }
)

source_fig.update_layout(
    mapbox_style='open-street-map'
)

In [ ]:
# @title
app.layout = html.Div([

    # ==========================================
    # TITLE
    # ==========================================

    html.H1(
        'New Zealand Earthquake Explorer',
        style={
            'textAlign': 'center'
        }
    ),


    # ==========================================
    # TWO MAPS SIDE BY SIDE
    # ==========================================

    html.Div([

        # --------------------------------------
        # LEFT MAP
        # --------------------------------------

        html.Div([

            html.H2(
                'Earthquake Sources',
                style={
                    'textAlign': 'center'
                }
            ),

            dcc.Graph(
                id='source-map',
                figure=source_fig,
                style={
                    'width': '100%',
                    'height': '750px'
                }
            )

        ], style={
            'width': '50%'
        }),


        # --------------------------------------
        # RIGHT MAP
        # --------------------------------------

        html.Div([

            html.H2(
                'Selected Earthquake',
                style={
                    'textAlign': 'center'
                }
            ),

            html.Div(
                id='event-info',
                children='Click an earthquake source above.',
                style={
                    'height': '80px',
                    'padding': '10px'
                }
            ),

            dcc.Graph(
                id='mmi-map',
                style={
                    'width': '100%',
                    'height': '750px'
                }
            )

        ], style={
            'width': '50%'
        })

    ],

    # ==========================================
    # SIDE BY SIDE
    # ==========================================

    style={
        'display': 'flex',
        'width': '100%',
        'gap': '10px'
    }),


    # ==========================================
    # OPTIONS
    # ==========================================

    html.Div([

        dcc.Checklist(
            id='map-options',

            options=[
                {
                    'label': ' Show source → MMI lines',
                    'value': 'lines'
                },
                {
                    'label': ' Show distance',
                    'value': 'distance'
                }
            ],

            value=[],

            inline=True
        )

    ], style={
        'textAlign': 'center',
        'margin': '15px'
    })

])

In [ ]:
# @title
@app.callback(
    Output('mmi-map', 'figure'),
    Output('event-info', 'children'),

    Input('source-map', 'clickData'),
    Input('map-options', 'value')
)
def update_mmi_map(clickData, options):

    # ==========================================
    # 1. CHƯA CLICK EARTHQUAKE
    # ==========================================

    if clickData is None:

        empty_fig = go.Figure()

        empty_fig.update_layout(
            mapbox_style='open-street-map',

            mapbox={
                'center': {
                    'lat': -41.3,
                    'lon': 172.5
                },
                'zoom': 4.5
            },

            title='Click an earthquake source'
        )

        return (
            empty_fig,
            'Click an earthquake source above.'
        )


    # ==========================================
    # 2. LẤY PUBLICID TỪ ĐIỂM ĐƯỢC CLICK
    # ==========================================

    publicid = clickData['points'][0]['customdata'][0]


    # ==========================================
    # 3. TÌM EARTHQUAKE TRONG LAYER_0
    # ==========================================

    event = layer_0[
        layer_0['publicid'] == publicid
    ].iloc[0]


    # ==========================================
    # 4. LẤY MMI REPORTS TỪ LAYER_1
    # ==========================================

    selected = layer_1[
        layer_1['publicid'] == publicid
    ].compute()


    # ==========================================
    # 5. REMOVE INVALID COORDINATES
    # ==========================================

    selected = selected.dropna(
        subset=[
            'report_latitude',
            'report_longitude'
        ]
    )


    # ==========================================
    # 6. NẾU EVENT KHÔNG CÓ MMI
    # ==========================================

    if selected.empty:

        empty_fig = go.Figure()

        empty_fig.update_layout(
            mapbox_style='open-street-map',

            mapbox={
                'center': {
                    'lat': event['event_latitude'],
                    'lon': event['event_longitude']
                },
                'zoom': 7
            },

            title=f'No MMI reports — {publicid}'
        )

        info = html.Div([

            html.B(
                f'Public ID: {publicid}'
            ),

            html.Br(),

            f"Mw: {event['Mw_model']}",

            html.Br(),

            f"Depth: {event['event_depth_km']} km",

            html.Br(),

            f"Fault style: {event['fault_style']}"
        ])

        return empty_fig, info


    # ==========================================
    # 7. MMI MAP
    # ==========================================

    fig = px.scatter_map(

        selected,

        lat='report_latitude',
        lon='report_longitude',

        color='reported_mmi',

        range_color=[0, 10],

        zoom=7,

        # CENTER = EARTHQUAKE SOURCE
        center={
            'lat': event['event_latitude'],
            'lon': event['event_longitude']
        },

        title=f'MMI Reports — {publicid}',

        # Dùng customdata để kiểm soát hover
        custom_data=[
            'reported_mmi',
            'Rhypo_km'
        ]
    )


    # ==========================================
    # 8. MMI HOVER
    # ==========================================

    fig.update_traces(

        hovertemplate=(
            '<b>MMI:</b> %{customdata[0]}<br>'
            '<b>Rhypo:</b> %{customdata[1]:.1f} km'
            '<extra></extra>'
        )
    )


    # ==========================================
    # 9. EARTHQUAKE SOURCE ★
    # ==========================================

    fig.add_trace(

        go.Scattermap(

            lat=[
                event['event_latitude']
            ],

            lon=[
                event['event_longitude']
            ],

            mode='markers',

            marker=dict(
                size=20,
                symbol='star'
            ),

            name='Earthquake Source',

            hovertemplate=(
                '<b>★ Earthquake Source</b><br>'
                f'Public ID: {publicid}<br>'
                f"Mw: {event['Mw_model']}<br>"
                f"Depth: {event['event_depth_km']} km<br>"
                f"Fault style: {event['fault_style']}"
                '<extra></extra>'
            )
        )
    )


    # ==========================================
    # 10. SOURCE → MMI LINES
    # ==========================================

    line_lat = []
    line_lon = []

    for _, row in selected.iterrows():

        line_lat.extend([
            event['event_latitude'],
            row['report_latitude'],
            None
        ])

        line_lon.extend([
            event['event_longitude'],
            row['report_longitude'],
            None
        ])


    fig.add_trace(

        go.Scattermap(

            lat=line_lat,
            lon=line_lon,

            mode='lines',

            name='Source → MMI',

            hoverinfo='skip',

            visible=(
                'lines' in (options or [])
            )
        )
    )


    # ==========================================
    # 11. RHYPO DISTANCE LABELS
    # ==========================================

    distance_lat = []
    distance_lon = []
    distance_text = []

    for _, row in selected.iterrows():

        if pd.notna(row['Rhypo_km']):

            # Vị trí giữa source và MMI point
            mid_lat = (
                event['event_latitude']
                + row['report_latitude']
            ) / 2

            mid_lon = (
                event['event_longitude']
                + row['report_longitude']
            ) / 2

            distance_lat.append(mid_lat)
            distance_lon.append(mid_lon)

            distance_text.append(
                f"{row['Rhypo_km']:.1f} km"
            )


    fig.add_trace(

        go.Scattermap(

            lat=distance_lat,

            lon=distance_lon,

            mode='text',

            text=distance_text,

            name='Hypocentral distance',

            hoverinfo='skip',

            visible=(
                'distance' in (options or [])
            )
        )
    )


    # ==========================================
    # 12. MAP SETTINGS
    # ==========================================

    fig.update_layout(

        mapbox_style='open-street-map',

        legend=dict(
            orientation='h'
        )
    )


    # ==========================================
    # 13. EVENT INFORMATION
    # ==========================================

    info = html.Div([

        html.B(
            f'★ {publicid}'
        ),

        html.Br(),

        f"Mw: {event['Mw_model']}",

        html.Br(),

        f"Depth: {event['event_depth_km']} km",

        html.Br(),

        f"Fault style: {event['fault_style']}",

        html.Br(),

        f"MMI reports: {len(selected)}"
    ])


    # ==========================================
    # 14. RETURN
    # ==========================================

    return fig, info

Map has show distance and line to connect

In [ ]:
# @title
from google.colab import output
import threading

def run_dash():
    app.run(
        debug=False,
        port=8050
    )

thread = threading.Thread(
    target=run_dash,
    daemon=True
)

thread.start()

output.serve_kernel_port_as_iframe(
    8050,
    width=1600,
    height=1600
)

<IPython.core.display.Javascript object>